# Step 4: Model Registration
Upload the trained model to S3 and register it in RHOAI Model Registry.

In [ ]:
import boto3
import requests
import json
import os

MODEL_PATH = os.environ.get("MODEL_PATH", "/tmp/model.joblib")
METRICS_PATH = os.environ.get("METRICS_PATH", "/tmp/metrics.json")
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://minio.pipeline-demo.svc:9000")
S3_BUCKET = os.environ.get("S3_BUCKET", "models")
MODEL_NAME = os.environ.get("MODEL_NAME", "loan-approval-classifier")
MODEL_VERSION = os.environ.get("MODEL_VERSION", "v1")
REGISTRY_URL = os.environ.get("REGISTRY_URL", "")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")

In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    verify=False,
)

try:
    s3.head_bucket(Bucket=S3_BUCKET)
except Exception:
    s3.create_bucket(Bucket=S3_BUCKET)
    print(f"Created bucket: {S3_BUCKET}")

s3_key = f"models/{MODEL_NAME}/{MODEL_VERSION}/model.joblib"
s3.upload_file(MODEL_PATH, S3_BUCKET, s3_key)
s3_uri = f"s3://{S3_BUCKET}/{s3_key}"
print(f"Model uploaded to {s3_uri}")

In [ ]:
if REGISTRY_URL:
    with open(METRICS_PATH) as f:
        metrics = json.load(f)

    resp = requests.post(
        f"{REGISTRY_URL}/api/model_registry/v1alpha3/registered_models",
        json={"name": MODEL_NAME, "description": "Loan approval classifier (RandomForest)"},
        verify=False,
    )
    model_id = resp.json().get("id")
    print(f"Registered model: {MODEL_NAME} (id={model_id})")

    resp = requests.post(
        f"{REGISTRY_URL}/api/model_registry/v1alpha3/model_versions",
        json={
            "name": MODEL_VERSION,
            "registeredModelId": model_id,
            "customProperties": {
                "s3_uri": {"string_value": s3_uri},
                "framework": {"string_value": "sklearn"},
                "accuracy": {"string_value": str(metrics.get("accuracy", ""))},
                "f1_score": {"string_value": str(metrics.get("f1_score", ""))},
            },
        },
        verify=False,
    )
    print(f"Registered version: {MODEL_VERSION}")
    print(f"Metrics: {metrics}")
else:
    print("REGISTRY_URL not set -- skipping Model Registry registration")
    print(f"Model is available at: {s3_uri}")